In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Clone the repository (Default is V2)
!git clone https://github.com/MIC-DKFZ/nnUNet.git

# 2. Install nnU-Net V2
%cd nnUNet
!pip install -e .
%cd ..

# 3. Install hiddenlayer (optional, often used for plotting)
!pip install --upgrade git+https://github.com/nanohanno/hiddenlayer.git@bugfix/get_trace_graph#egg=hiddenlayer

Cloning into 'nnUNet'...
remote: Enumerating objects: 14750, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 14750 (delta 110), reused 78 (delta 58), pack-reused 14598 (from 2)
Receiving objects: 100% (14750/14750), 8.98 MiB | 18.84 MiB/s, done.
Resolving deltas: 100% (11085/11085), done.
/content/nnUNet
Obtaining file:///content/nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.3 MB/s eta 0:00:00
  Insta

/content
  Cloning https://github.com/nanohanno/hiddenlayer.git (to revision bugfix/get_trace_graph) to /tmp/pip-install-oby8vfpi/hiddenlayer_5b0f388d91ad4a108a7b623f8003dcd4
  Running command git clone --filter=blob:none --quiet https://github.com/nanohanno/hiddenlayer.git /tmp/pip-install-oby8vfpi/hiddenlayer_5b0f388d91ad4a108a7b623f8003dcd4
  Running command git checkout -b bugfix/get_trace_graph --track origin/bugfix/get_trace_graph
  Switched to a new branch 'bugfix/get_trace_graph'
  Branch 'bugfix/get_trace_graph' set up to track remote branch 'bugfix/get_trace_graph' from 'origin'.
  Resolved https://github.com/nanohanno/hiddenlayer.git to commit 321c5d934c6ed3a7e8d330bc4adcf7764bc440c3
  Preparing metadata (setup.py) ... done
  Created wheel for hiddenlayer: filename=hiddenlayer-0.2-py3-none-any.whl size=19733 sha256=97387e6a90bd9f7d488ddc2a8216a0f74b038c4f68ea2425b214c29837e3b3dd
  Stored in directory: /tmp/pip-ephem-wheel-cache-08gl60l_/wheels/ab/fd/da/37347b8fe52a978326cf2d

In [ ]:
# 1. Download the model zip file
# I am using a generic name 'new_model.zip', you can rename it to whatever you like
# COPY the download link from the "Files" section of your Zenodo page
!wget -O new_model.zip "https://zenodo.org/records/11582627/files/Dataset002_BRATS19.zip?download=1"
# NOTE: I assumed the filename above based on the record description (BraTS-GLI).
# If the download fails, please paste the specific file URL from the Zenodo page.

--2026-03-28 11:54:30--  https://zenodo.org/records/11582627/files/Dataset002_BRATS19.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 188.185.48.75, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1155915349 (1.1G) [application/octet-stream]
Saving to: ‘new_model.zip’

new_model.zip       100%[===================>]   1.08G  12.7MB/s    in 90s     

2026-03-28 11:56:01 (12.2 MB/s) - ‘new_model.zip’ saved [1155915349/1155915349]



In [ ]:
import os

# 1. Make sure the folders exist
os.makedirs('./nnUNet_raw', exist_ok=True)
os.makedirs('./nnUNet_preprocessed', exist_ok=True)
os.makedirs('./nnUNet_results', exist_ok=True)

# 2. Download the model (if you haven't already)
# (Skipping download if 'new_model.zip' already exists to save time)
if not os.path.exists('new_model.zip'):
    !wget -O new_model.zip "https://zenodo.org/records/11582627/files/Dataset002_BRATS19.zip?download=1"

# 3. Install with EXPLICIT environment variables
# We use 'env' at the start of the command to guarantee nnU-Net sees the paths
!env nnUNet_raw="./nnUNet_raw" \
     nnUNet_preprocessed="./nnUNet_preprocessed" \
     nnUNet_results="./nnUNet_results" \
     nnUNetv2_install_pretrained_model_from_zip new_model.zip

In [ ]:
import os
import zipfile
import shutil
import tempfile

# Paths
zip_file = "new_model.zip"
results_folder = "./nnUNet_results"
expected_dataset_id = "226"
final_folder_name = "Dataset226_BraTS2024"

print(f"Checking {zip_file}...")

if not os.path.exists(zip_file):
    print("❌ Error: Zip file not found. Please re-download it.")
else:
    try:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            file_list = zip_ref.namelist()
            top_folder = file_list[0].split('/')[0]
            print(f"   Zip top-level folder appears to be: '{top_folder}'")

            os.makedirs(results_folder, exist_ok=True)

            # Case 1: Zip already contains Dataset226_*
            if top_folder.startswith(f"Dataset{expected_dataset_id}"):
                print("   Structure looks correct. Extracting directly...")
                zip_ref.extractall(results_folder)

                extracted_path = os.path.join(results_folder, top_folder)
                final_path = os.path.join(results_folder, final_folder_name)

                if top_folder != final_folder_name:
                    print(f"   Renaming '{top_folder}' → '{final_folder_name}'")
                    if os.path.exists(final_path):
                        shutil.rmtree(final_path)
                    os.rename(extracted_path, final_path)

            # Case 2: Zip contains a *different* DatasetXXX folder
            else:
                print("   Structure mismatch. Extracting and renaming dataset folder...")

                with tempfile.TemporaryDirectory() as tmpdir:
                    zip_ref.extractall(tmpdir)

                    # Find the dataset folder inside temp
                    subfolders = [
                        f for f in os.listdir(tmpdir)
                        if os.path.isdir(os.path.join(tmpdir, f)) and f.startswith("Dataset")
                    ]

                    if len(subfolders) != 1:
                        raise RuntimeError(f"Expected exactly one Dataset folder, found: {subfolders}")

                    src = os.path.join(tmpdir, subfolders[0])
                    dst = os.path.join(results_folder, final_folder_name)

                    if os.path.exists(dst):
                        shutil.rmtree(dst)

                    shutil.move(src, dst)

        print("✅ Extraction and renaming complete.")

        # Final verification
        print("\nContents of nnUNet_results:")
        print(os.listdir(results_folder))

    except zipfile.BadZipFile:
        print("❌ Error: The zip file is corrupted. Please try downloading it again.")


Checking new_model.zip...
   Zip top-level folder appears to be: 'Dataset002_BRATS19'
   Structure mismatch. Extracting and renaming dataset folder...
✅ Extraction and renaming complete.

Contents of nnUNet_results:
['Dataset002_BRATS19', 'Dataset226_BraTS2024']


In [ ]:
# # data_dir = "/content/drive/MyDrive/brain_tumor_segmentation/Datasets/brats23_pediatric"
# json_list = "/content/drive/MyDrive/brain_tumor_segmentation/brats2023_pediatrics_5fold.json"

In [ ]:
import json

# Your new paths
data_dir = "/content/drive/MyDrive/brain_tumor_segmentation/Datasets/brats23_pediatric"
json_list = "/content/drive/MyDrive/brain_tumor_segmentation/brats2023_pediatrics_5fold.json"

with open(json_list, 'r') as f:
    input_data = json.load(f)

print(f"Successfully loaded JSON. Found {len(input_data.get('training', []))} total cases.")

Successfully loaded JSON. Found 99 total cases.


In [ ]:
input_data

{'training': [{'fold': 0,
   'image': ['ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00002-000/BraTS-PED-00002-000-t2f.nii.gz',
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00002-000/BraTS-PED-00002-000-t1c.nii.gz',
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00002-000/BraTS-PED-00002-000-t1n.nii.gz',
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00002-000/BraTS-PED-00002-000-t2w.nii.gz'],
   'label': 'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00002-000/BraTS-PED-00002-000-seg.nii.gz'},
  {'fold': 0,
   'image': ['ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00008-000/BraTS-PED-00008-000-t2f.nii.gz',
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00008-000/BraTS-PED-00008-000-t1c.nii.gz',
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00008-000/BraTS-PED-00008-000-t1n.nii.gz',
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/BraTS-PED-00008-000/Br

In [ ]:
import os
import json
import shutil
from glob import glob

# --- CONFIGURATION ---
TARGET_FOLD = [0, 1, 2, 3, 4]   # <--- SET THIS to a list of the folds you want to run
inference_dir = '/content/test_input'
dataset_json_path = '/content/nnUNet_results/Dataset226_BraTS2024/nnUNetTrainer__nnUNetPlans__3d_fullres/dataset.json'

# SETUP PATHS
if os.path.exists(inference_dir):
    shutil.rmtree(inference_dir) # Clean up previous run
os.makedirs(inference_dir, exist_ok=True)

# DETECT CHANNEL MAPPING
channel_map = {}
if os.path.exists(dataset_json_path):
    with open(dataset_json_path, 'r') as f:
        d = json.load(f)
        channel_names = d.get('channel_names', {})
        for k, v in channel_names.items():
            channel_map[v.lower()] = int(k)
else:
    print("⚠️ dataset.json not found. Using text-based fallback.")

# PREPARE FILES
print(f"\nStarting File Preparation for FOLDS: {TARGET_FOLD}...")
count = 0

for item in input_data['training']:
    # --- CHANGED: Check if the item's fold is in your target list ---
    if item['fold'] not in TARGET_FOLD:
        continue

    files = item['image']
    basename = os.path.basename(files[0])
    case_id = basename.rsplit('-', 1)[0]

    print(f"Processing {case_id} (Fold {item['fold']})...")
    count += 1

    for file_path in files:
        full_src_path = os.path.join(data_dir, file_path)

        if not os.path.exists(full_src_path):
            print(f"  ❌ File not found: {full_src_path}")
            continue

        filename_lower = file_path.lower()
        suffix = None

        # Mapping Logic
        if 't1c' in filename_lower: suffix = '_0000'
        elif 't1n' in filename_lower: suffix = '_0001'
        elif 't2f' in filename_lower or 'flair' in filename_lower: suffix = '_0002'
        elif 't2w' in filename_lower: suffix = '_0003'

        if suffix:
            dst_path = os.path.join(inference_dir, f"{case_id}{suffix}.nii.gz")
            shutil.copy(full_src_path, dst_path)

print(f"\n✅ Complete. Prepared {count} total cases in {inference_dir}")


Starting File Preparation for FOLDS: [0, 1, 2, 3, 4]...
Processing BraTS-PED-00002-000 (Fold 0)...
Processing BraTS-PED-00008-000 (Fold 0)...
Processing BraTS-PED-00023-000 (Fold 0)...
Processing BraTS-PED-00025-000 (Fold 0)...
Processing BraTS-PED-00040-000 (Fold 0)...
Processing BraTS-PED-00046-000 (Fold 0)...
Processing BraTS-PED-00051-000 (Fold 0)...
Processing BraTS-PED-00061-000 (Fold 0)...
Processing BraTS-PED-00079-000 (Fold 0)...
Processing BraTS-PED-00081-000 (Fold 0)...
Processing BraTS-PED-00083-000 (Fold 0)...
Processing BraTS-PED-00086-000 (Fold 0)...
Processing BraTS-PED-00109-000 (Fold 0)...
Processing BraTS-PED-00111-000 (Fold 0)...
Processing BraTS-PED-00120-000 (Fold 0)...
Processing BraTS-PED-00123-000 (Fold 0)...
Processing BraTS-PED-00131-000 (Fold 0)...
Processing BraTS-PED-00135-000 (Fold 0)...
Processing BraTS-PED-00143-000 (Fold 0)...
Processing BraTS-PED-00170-000 (Fold 0)...
Processing BraTS-PED-00009-000 (Fold 1)...
Processing BraTS-PED-00021-000 (Fold 1).

In [ ]:
# This searches for the typo and replaces it with the correct syntax
!sed -i 's/queue = manager().Queue(maxsize=1)/queue = manager.Queue(maxsize=1)/g' /content/nnUNet/nnunetv2/inference/data_iterators.py
!sed -i 's/queue = Manager().Queue(maxsize=1)/queue = manager.Queue(maxsize=1)/g' /content/nnUNet/nnunetv2/inference/data_iterators.py

In [ ]:
# !env nnUNet_raw="./nnUNet_raw" \
#      nnUNet_preprocessed="./nnUNet_preprocessed" \
#      nnUNet_results="./nnUNet_results" \
#      nnUNetv2_predict -i /content/test_input -o /content/test_output -d 226 -c 3d_fullres -f all

!env nnUNet_raw="./nnUNet_raw" \
     nnUNet_preprocessed="./nnUNet_preprocessed" \
     nnUNet_results="./nnUNet_results" \
     nnUNetv2_predict -i /content/test_input \
                      -o /content/test_output \
                      -d 226 \
                      -c 3d_fullres \
                      -f 0 1 2 3 4  # <--- CHANGED back to 'all'


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

/content/nnUNet/nnunetv2/utilities/plans_handling/plans_handler.py:37: UserWarning: Detected old nnU-Net plans format. Attempting to reconstruct network architecture parameters. If this fails, rerun nnUNetv2_plan_experiment for your dataset. If you use a custom architecture, please downgrade nnU-Net to the version you implemented this or update your implementation + plans.
  warnings.warn("Detected old nnU-Net plans format. Attempting to reconstruct network architecture "
There are 99 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
Ther

In [ ]:
import os
import shutil
import nibabel as nib
import numpy as np

# --- CONFIGURATION ---
TARGET_FOLD = [0, 1, 2, 3, 4]  # <--- Keep this consistent with the list above
gt_folder = '/content/test_gt'
pred_folder = '/content/test_output'

os.makedirs(gt_folder, exist_ok=True)

print(f"Preparing Ground Truth for evaluation (FOLDS: {TARGET_FOLD})...")

for item in input_data['training']:
    # --- CHANGED: Check if the item's fold is in your target list ---
    if item['fold'] not in TARGET_FOLD:
        continue

    # Identify the Case ID
    files = item['image']
    basename = os.path.basename(files[0])
    case_id = basename.rsplit('-', 1)[0]

    gt_source = os.path.join(data_dir, item['label'])

    # Check if we actually have a prediction for this case
    pred_path = os.path.join(pred_folder, f"{case_id}.nii.gz")
    if not os.path.exists(pred_path):
        print(f"Skipping {case_id} (No prediction found)")
        continue

    # Copy and Fix Label
    if os.path.exists(gt_source):
        target_path = os.path.join(gt_folder, f"{case_id}.nii.gz")

        # Load NIfTI
        nimg = nib.load(gt_source)
        data = nimg.get_fdata()

        # CHECK FOR LABEL 4 (BraTS standard) AND CONVERT TO 3 (nnU-Net standard)
        if np.any(data == 4):
            print(f"  Fixing labels for {case_id} (4 -> 3)")
            data[data == 4] = 3
            new_nimg = nib.Nifti1Image(data, nimg.affine, nimg.header)
            nib.save(new_nimg, target_path)
        else:
            shutil.copy(gt_source, target_path)

    else:
        print(f"Warning: GT file not found: {gt_source}")

print("Ground Truth preparation complete.")

Preparing Ground Truth for evaluation (FOLDS: [0, 1, 2, 3, 4])...
Ground Truth preparation complete.


In [ ]:
import os
import nibabel as nib
import numpy as np

gt_folder = '/content/test_gt'

# --- CONFIGURATION: SET THIS BASED ON YOUR DIAGNOSIS ---
# Set to True if Model outputs 4, but GT has 3.
CONVERT_3_TO_4 = True

# Set to True if Model outputs 3, but GT has 4.
CONVERT_4_TO_3 = False
# -------------------------------------------------------

print(f"Fixing labels in {gt_folder}...")

for filename in os.listdir(gt_folder):
    if not filename.endswith('.nii.gz'): continue

    path = os.path.join(gt_folder, filename)
    nimg = nib.load(path)
    data = nimg.get_fdata()

    original_values = np.unique(data).astype(int)
    modified = False

    if CONVERT_4_TO_3 and 4 in original_values:
        print(f"  {filename}: Converting 4 -> 3")
        data[data == 4] = 3
        modified = True

    if CONVERT_3_TO_4 and 3 in original_values:
        print(f"  {filename}: Converting 3 -> 4")
        data[data == 3] = 4
        modified = True

    if modified:
        new_nimg = nib.Nifti1Image(data, nimg.affine, nimg.header)
        nib.save(new_nimg, path)

print("Done. Now re-run the evaluation command.")

Fixing labels in /content/test_gt...
  BraTS-PED-00050-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00116-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00083-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00108-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00080-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00135-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00124-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00064-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00134-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00113-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00125-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00118-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00067-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00057-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00109-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00010-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00081-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00104-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00006-000.nii.gz: Converting 3 -> 4
  BraTS-PED-00096-000.nii.gz: Converting 3 -> 4
  B

In [ ]:
!pip install medpy nibabel pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for medpy: filename=MedPy-0.5.2-py3-none-any.whl size=224710 sha256=8f0e0cfd522c6ed0518e2060a7a71fad73d3b3911beee3ff7b34f3dfd3042999
  Stored in directory: /root/.cache/pip/wheels/89/5a/f8/b3def53b9c2133d2f8698ea2173bb5df63bd8e761ce8e9aec9
Successfully built medpy


In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from medpy import metric

# --- CONFIGURATION ---
gt_folder = '/content/test_gt'
pred_folder = '/content/test_output'

# Define BraTS Regions
# Key = Region Name
# Value = List of labels to include in this region
regions = {
    'ET (Enhancing Tumor)': [4],       # Label 4
    'TC (Tumor Core)':      [1, 4],    # Label 1 + 4
    'WT (Whole Tumor)':     [1, 2, 4]  # Label 1 + 2 + 4
}

results = []

print(f"Starting BraTS Evaluation on {len(os.listdir(pred_folder))} files...")

for filename in sorted(os.listdir(pred_folder)):
    if not filename.endswith('.nii.gz') or filename == 'summary.json':
        continue

    pred_path = os.path.join(pred_folder, filename)
    gt_path = os.path.join(gt_folder, filename)

    if not os.path.exists(gt_path):
        print(f"Skipping {filename}: No Ground Truth found.")
        continue

    # Load Data
    pred_data = nib.load(pred_path).get_fdata().astype(int)
    gt_data = nib.load(gt_path).get_fdata().astype(int)

    # Check if header matches (spacing is important for HD95)
    header = nib.load(pred_path).header
    zoom = header.get_zooms() # Voxel spacing (e.g. 1.0mm x 1.0mm x 1.0mm)

    row = {'Case': filename}

    for region_name, labels in regions.items():
        # Create Boolean Masks for the Region
        # numpy.isin checks if values are in our list [1, 4] etc.
        mask_pred = np.isin(pred_data, labels)
        mask_gt = np.isin(gt_data, labels)

        # --- DICE CALCULATION ---
        if mask_pred.sum() == 0 and mask_gt.sum() == 0:
            dice = 1.0 # Both empty matches
        elif mask_pred.sum() > 0 and mask_gt.sum() == 0:
            dice = 0.0 # False Positive
        elif mask_pred.sum() == 0 and mask_gt.sum() > 0:
            dice = 0.0 # False Negative
        else:
            dice = metric.dc(mask_pred, mask_gt)

        # --- HD95 CALCULATION ---
        # HD95 fails if either mask is empty. We handle that here.
        if mask_pred.sum() > 0 and mask_gt.sum() > 0:
            try:
                # voxelspacing argument ensures we get millimeters, not pixels
                hd95 = metric.binary.hd95(mask_pred, mask_gt, voxelspacing=zoom)
            except RuntimeError:
                hd95 = np.nan # Can fail on weird geometries
        elif mask_pred.sum() == 0 and mask_gt.sum() == 0:
            hd95 = 0.0 # Perfect match (both empty)
        else:
            # If one is empty and other isn't, distance is technically "max distance"
            # often set to 373 (max dist in brain) or NaN. We'll use 373 for penalty.
            hd95 = 373.0

        row[f'{region_name} Dice'] = dice
        row[f'{region_name} HD95'] = hd95

    results.append(row)
    print(f"Processed {filename}")

# --- DISPLAY RESULTS ---
if results:
    df = pd.DataFrame(results)

    # Calculate Means
    mean_row = df.mean(numeric_only=True)
    mean_row['Case'] = 'AVERAGE'
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

    # Reorder columns
    cols = ['Case'] + [c for c in df.columns if c != 'Case']
    df = df[cols]

    print("\n" + "="*50)
    print("FINAL BRATS BENCHMARK RESULTS")
    print("="*50)
    print(df.to_string())

    # Save to CSV
    df.to_csv('/content/brats_benchmark_results.csv', index=False)
    print("\nSaved detailed results to /content/brats_benchmark_results.csv")
else:
    print("No cases processed.")

Starting BraTS Evaluation on 102 files...
Processed BraTS-PED-00002-000.nii.gz
Processed BraTS-PED-00003-000.nii.gz
Processed BraTS-PED-00004-000.nii.gz
Processed BraTS-PED-00006-000.nii.gz
Processed BraTS-PED-00008-000.nii.gz
Processed BraTS-PED-00009-000.nii.gz
Processed BraTS-PED-00010-000.nii.gz
Processed BraTS-PED-00019-000.nii.gz
Processed BraTS-PED-00020-000.nii.gz
Processed BraTS-PED-00021-000.nii.gz
Processed BraTS-PED-00023-000.nii.gz
Processed BraTS-PED-00024-000.nii.gz
Processed BraTS-PED-00025-000.nii.gz
Processed BraTS-PED-00026-000.nii.gz
Processed BraTS-PED-00032-000.nii.gz
Processed BraTS-PED-00036-000.nii.gz
Processed BraTS-PED-00037-000.nii.gz
Processed BraTS-PED-00039-000.nii.gz
Processed BraTS-PED-00040-000.nii.gz
Processed BraTS-PED-00041-000.nii.gz
Processed BraTS-PED-00042-000.nii.gz
Processed BraTS-PED-00044-000.nii.gz
Processed BraTS-PED-00046-000.nii.gz
Processed BraTS-PED-00047-000.nii.gz
Processed BraTS-PED-00048-000.nii.gz
Processed BraTS-PED-00050-000.nii